## Import necessary packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import anndata as ad
import pandas as pd
from scipy.sparse import csr_matrix
from CellPLM.utils import set_seed
from CellPLM.pipeline.continual_pretraining import ContinualPretrainingPipeline, ContinualPretrainingDefaultPipelineConfig
import scanpy as sc
import matplotlib.pyplot as plt
# import rapids_singlecell as rsc  # For faster evaluation, we recommend the installation of rapids_singlecell.

## Specify important parameters before getting started

In [ ]:
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'

## Load Downstream Dataset

In [ ]:
set_seed(42)

dataset='ATAA'

data = ad.read_h5ad('/media/rokny/DATA2/Sally/data/scRNA-seq/ATAA/gse155468.h5ad')
data.obs_names_make_unique()

data.obs['split'] = 'train'

In [ ]:
# normalize each cell to target sum (e.g., 1e4 = CP10k)
sc.pp.normalize_total(data, target_sum=1e4)   # in-place on adata.X

# log1p transform (natural log)
sc.pp.log1p(data)

## Set up the pipeline

In [ ]:
import json

config_path = './ckpt/20231027_85M.config.json'

with open(config_path, 'r') as f:
    model_config = json.load(f)

pipeline_config = ContinualPretrainingDefaultPipelineConfig.copy()

pipeline_config, model_config

In [ ]:
model_config['objective'] = 'recon'

In [ ]:
# Convert gene symbols to Ensembl IDs
import mygene

mg = mygene.MyGeneInfo()

gene_symbols = data.var_names.tolist()

out = mg.querymany(
    gene_symbols,
    scopes="symbol",  
    fields="ensembl.gene",
    species="human"
)

df = pd.DataFrame(out)

# Some genes return multiple Ensembl IDs; keep the first one
df["ensembl_id"] = df["ensembl"].apply(
    lambda x: x[0]["gene"] if isinstance(x, list) else (x["gene"] if isinstance(x, dict) else None)
)

# Handle missing Ensembl IDs (replace None with original gene symbol or drop them)
df["ensembl_id"].fillna(df["query"], inplace=True)

# Build mapping dictionary: {symbol -> ensembl_id}
mapping = df.set_index("query")["ensembl_id"].to_dict()

# Update var_names with Ensembl IDs
data.var["gene_symbol"] = data.var_names
data.var_names = [mapping.get(g, g) for g in data.var_names]

# Check for duplicates after gene name conversion
if not data.var_names.is_unique:
    keep = ~data.var_names.duplicated(keep='first')
    data = data[:, keep].copy()

In [ ]:
from scipy import sparse

def align_adata_to_model_genes(adata, model_genes):
    adata = adata.copy()

    current_genes = list(adata.var_names)
    missing_genes = [g for g in model_genes if g not in current_genes]
    print(f"Missing {len(missing_genes)} / {len(model_genes)} genes.")

    # create zero columns for missing genes
    if missing_genes:
        n_cells = adata.n_obs
        n_missing = len(missing_genes)
        if sparse.issparse(adata.X):
            zeros = sparse.csr_matrix((n_cells, n_missing), dtype=adata.X.dtype)
            newX = sparse.hstack([adata.X, zeros]).tocsr()
        else:
            zeros = np.zeros((n_cells, n_missing), dtype=adata.X.dtype)
            newX = np.hstack([adata.X, zeros])

        # build new .var dataframe
        new_var = pd.concat([adata.var, pd.DataFrame(index=missing_genes)])
        assert newX.shape[1] == new_var.shape[0]

        # rebuild AnnData safely
        adata = ad.AnnData(
            X=newX,
            obs=adata.obs.copy(),
            var=new_var
        )

    # reorder genes to match model
    adata = adata[:, model_genes].copy()

    # sanity check
    assert list(adata.var_names) == list(model_genes)
    print(f"Final shape: {adata.shape}")
    return adata

model_genes = model_config["gene_list"]
data = align_adata_to_model_genes(data, model_genes)

In [ ]:
data

In [ ]:
if 'batch' not in data.obs.columns:
    data.obs['batch'] = 'all'

batch_gene_list = {}

batches = data.obs['batch'].astype(str).unique().tolist()
cfg_genes = set(model_config['gene_list'])
eligible = [g for g in data.var_names if g in cfg_genes]
batch_gene_list = {b: eligible for b in batches}

In [ ]:
pipeline = ContinualPretrainingPipeline(pretrain_prefix=PRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                 overwrite_config=model_config,
                                 pretrain_directory='../ckpt')
pipeline.model

## Do continual pretraining

In [ ]:
pipeline.fit(data,
            pipeline_config,
            batch_gene_list=batch_gene_list,
            device=DEVICE)

## Save checkpoint

In [ ]:
import json
import torch

checkpoint = {
    'model_state_dict': pipeline.model.state_dict(),  # the model's weights
    'config': model_config  # model configuration
}

checkpoint_path = f'./ckpt/continualpretrain_{dataset}.best.ckpt'
config_path = f'./ckpt/continualpretrain_{dataset}.config.json'

# Save the checkpoint (model weights, optimizer state)
torch.save(checkpoint, checkpoint_path)

# Save the model config
with open(config_path, 'w') as f:
    json.dump(model_config, f) 

In [ ]:
import json
import torch

# Load the first config (the one with missing information)
with open(f'./ckpt/continualpretrain_{dataset}.config.json', 'r') as f1:
    config_1 = json.load(f1)

# Load the second config (the one that contains the missing information)
with open('./ckpt/20231027_85M.config.json', 'r') as f2:
    config_2 = json.load(f2)

# Merge the missing information from config_2 into config_1
# For each key in config_2, check if it's missing in config_1, and if so, add it.
for key, value in config_2.items():
    if key not in config_1:
        config_1[key] = value

model_config = config_1


In [ ]:
checkpoint = {
    'model_state_dict': pipeline.model.state_dict(),  # the model's weights
    'config': model_config  # model configuration
}

checkpoint_path = f'./ckpt/continualpretrain_{dataset}.best.ckpt'
config_path = f'./ckpt/continualpretrain_{dataset}.config.json'

# Save the checkpoint (model weights, optimizer state)
torch.save(checkpoint, checkpoint_path)

# Save the model config
with open(config_path, 'w') as f:
    json.dump(model_config, f) 

## Extract embeddings

In [ ]:
from CellPLM.pipeline.cell_embedding import CellEmbeddingPipeline

In [ ]:
CONTINUALPRETRAIN_VERSION = f'continualpretrain_{dataset}'

In [ ]:
pipeline = CellEmbeddingPipeline(pretrain_prefix=CONTINUALPRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                 pretrain_directory='../ckpt')
pipeline.model

In [ ]:
embedding = pipeline.predict(data, # An AnnData object
                device=DEVICE) # Specify a gpu or cpu for model inference

data.obsm['emb'] = embedding.cpu().numpy()

## Save embeddings

In [ ]:
output_path = f'./embeddings/{dataset}_with_continualpretrain_embeddings.h5ad'

data.write_h5ad(output_path)

print(f"Embeddings saved to {output_path} in obsm['emb']")

## UMAP Visualisation of Embeddings

In [ ]:
# Set a seed for reproducibility

import os
import torch
import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Read the AnnData objects
data = sc.read_h5ad(f"./embeddings/{dataset}_with_continualpretrain_embeddings.h5ad")

In [ ]:
data

## Evaluation and Inference

In [ ]:
sc.pp.neighbors(data, use_rep='emb', random_state=seed)
sc.tl.umap(data, random_state=seed)
plt.rcParams['figure.figsize'] = (6, 6)
sc.pl.umap(data, color='celltype', palette='Paired', title='CellPLM', show=False)
ax = plt.gca()
handles, labels = ax.get_legend_handles_labels()
plt.legend(
    handles, labels,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    ncol=1, 
    fontsize='small',
    frameon=False
)
file_path = f'./figures/umap_continual_pretrain_clustering_{dataset}.svg'
plt.savefig(file_path, dpi=500, bbox_inches='tight')
plt.show()

## Clustering

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.decomposition import PCA


def clustering(adata, n_clusters=7, key='emb', method='leiden', start=0.1, end=3.0, increment=0.01):
    """\
    Spatial clustering based the learned representation.

    Returns
    -------
    None.

    """
    
    pca = PCA(n_components=20, random_state=seed) 
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding
    
    if method == 'leiden':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['louvain'] 
       
    

def extract_top_value(map_matrix, retain_percent = 0.1): 
    '''\
    Filter out cells with low mapping probability

    Parameters
    ----------
    map_matrix : array
        Mapped matrix with m spots and n cells.
    retain_percent : float, optional
        The percentage of cells to retain. The default is 0.1.

    Returns
    -------
    output : array
        Filtered mapped matrix.

    '''

    #retain top 1% values for each spot
    top_k  = retain_percent * map_matrix.shape[1]
    output = map_matrix * (np.argsort(np.argsort(map_matrix)) >= map_matrix.shape[1] - top_k)
    
    return output 
    
def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number
    
    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.    
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float 
        The end value for searching.
    increment : float
        The step size to increase.
        
    Returns
    -------
    res : float
        Resolution.
        
    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep, random_state=seed)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique()) 
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!." 
       
    return res    


In [ ]:
# Run Leiden clustering 

n_clusters = 11
tool='leiden'

clustering(data, n_clusters, key='emb', method=tool, start=0.1, end=0.2, increment=0.01)

In [ ]:
labels = data.obs['leiden'].astype(int)

## ARI, NMI & Silhouette Scores

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

ari_score = adjusted_rand_score(data.obs['leiden'].to_numpy(), data.obs['celltype'].to_numpy())
nmi_score = normalized_mutual_info_score(data.obs['leiden'].to_numpy(), data.obs['celltype'].to_numpy())
sil_score = silhouette_score(data.obsm['emb'], data.obs['leiden'].astype(int))

print(f"'ari': {ari_score}, 'nmi': {nmi_score}, 'sil': {sil_score}")

## Save results to npz file

In [ ]:
Model_name='cellplm'
step='continualpretrain'

In [ ]:
import numpy as np

ARI, NMI, SIL = float(ari_score), float(nmi_score), float(sil_score)

np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_clusters_{dataset}.npz",
    labels=labels,      
    embeddings=data.obsm['emb'],
    ARI=ARI, NMI=NMI, SIL=SIL,
)